# Load model data

In [38]:
from dotenv import load_dotenv
import os

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
GROQ_MODEL = "llama-3.1-8b-instant"
GEMINI_API_KEY = os.getenv("GEMINI_API")
GEMINI_MODEL = "gemma-3-4b-it"

In [39]:
# Constants and configuration
CHUNK_SIZE = 25
OUTPUT_FOLDER = "../data/scored_data"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

In [40]:
import re
import numpy as np
import pandas as pd

def parse_scores(response_text, expected_count):
    """
    Extracts all digits/floats from model response, 
    rounds answers to the nearest integer, and clamps them to 0-1.
    Returns exactly 'expected_count' results (pad with None if missing).
    """
    if not response_text:
        return [pd.NA] * expected_count
        
    # Find sequence of numbers
    matches = re.findall(r"\b\d+(?:\.\d+)?\b", str(response_text))
    
    scores = []
    for match in matches:
        try:
            val = float(match)
            val = int(round(val))
            val = max(0, min(1, val))
            scores.append(val)
        except ValueError:
            pass
            
    # Normalize length
    if len(scores) < expected_count:
        scores.extend([pd.NA] * (expected_count - len(scores)))
    elif len(scores) > expected_count:
        scores = scores[:expected_count]
        
    return scores

def build_prompt(reviews_chunk):
    prompt = (
        "Analyze the sentiment of the following product reviews. "
        "For each review, provide a single integer score 0 or 1, "
        "where 0 is negative and 1 is positive. No decimals, no halves. "
        "Provide ONLY the numeric scores separated by commas in the exact same order as the reviews. "
        "Do not include any text, explanations, or headings.\n\n"
    )
    for i, review in enumerate(reviews_chunk):
        # Trim excessively long reviews to preserve input tokens (e.g. max 1000 chars)
        trimmed_review = str(review)[:1000]
        prompt += f"Review {i+1}:\n{trimmed_review}\n\n"
    return prompt

# Load datasets

In [41]:
import pandas as pd

all_cleaned_df = pd.read_csv("../data/cleaned_data/cleaned_reviews_all_flags.csv")
no_lemma_no_stopwords_df = pd.read_csv("../data/cleaned_data/cleaned_reviews_no_lemma_no_stopwords.csv")
no_special_no_lowercase_df = pd.read_csv("../data/cleaned_data/cleaned_reviews_no_special_no_lowercase.csv")

In [42]:
def normalize_socres(score):
    if score == 1.0 or score == 2.0:
        return 0
    elif score == 4.0 or score ==5.0:
        return 1

In [43]:
all_cleaned_df = all_cleaned_df[all_cleaned_df["rating"] != 3]
no_lemma_no_stopwords_df = no_lemma_no_stopwords_df[no_lemma_no_stopwords_df["rating"] != 3]
no_special_no_lowercase_df = no_special_no_lowercase_df[no_special_no_lowercase_df["rating"] != 3]

In [44]:
all_cleaned_df["rating"] = all_cleaned_df["rating"].apply(lambda x: normalize_socres(x))
no_lemma_no_stopwords_df["rating"] = no_lemma_no_stopwords_df["rating"].apply(lambda x : normalize_socres(x))
no_special_no_lowercase_df["rating"] = no_special_no_lowercase_df["rating"].apply(lambda x : normalize_socres(x))

# Get ground truth from models

In [45]:
import time
from groq import Groq
from google import genai
from google.genai import types

# Setup clients
groq_client = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None
gemini_client = genai.Client(api_key=GEMINI_API_KEY) if GEMINI_API_KEY else None

def get_groq_scores(reviews_chunk):
    if not groq_client:
        return [pd.NA] * len(reviews_chunk)
        
    prompt = build_prompt(reviews_chunk)
    
    for attempt in range(3):
        try:
            response = groq_client.chat.completions.create(
                model=GROQ_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0
            )
            result_text = response.choices[0].message.content
            scores = parse_scores(result_text, len(reviews_chunk))
            
            # If all are NA, try again; otherwise success
            if not all(pd.isna(x) for x in scores):
                return scores
        except Exception as e:
            err_msg = str(e).lower()
            if "429" in err_msg or "rate limit" in err_msg:
                print(f"Groq Rate Limit (429) hit. Waiting 60 seconds... (Attempt {attempt+1})")
                time.sleep(60)
            else:
                print(f"Groq error: {e}. Waiting 2s... (Attempt {attempt+1})")
                time.sleep(2)
            
    return [pd.NA] * len(reviews_chunk)

def get_gemini_scores(reviews_chunk):
    if not gemini_client:
        return [pd.NA] * len(reviews_chunk)
        
    prompt = build_prompt(reviews_chunk)
    for attempt in range(3):
        try:
            response = gemini_client.models.generate_content(
                model=GEMINI_MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(
                    temperature=0.0
                )
            )
            scores = parse_scores(response.text, len(reviews_chunk))
            
            if not all(pd.isna(x) for x in scores):
                return scores
        except Exception as e:
            err_msg = str(e).lower()
            if "429" in err_msg or "rate limit" in err_msg:
                print(f"Gemini Rate Limit (429) hit. Waiting 60 seconds... (Attempt {attempt+1})")
                time.sleep(60)
            else:
                print(f"Gemini error: {e}. Waiting 2s... (Attempt {attempt+1})")
                time.sleep(2)
            
    return [pd.NA] * len(reviews_chunk)

In [46]:
from tqdm.auto import tqdm
import os

# Registry of datasets to process
datasets_to_run = {
    "all_flags": ("cleaned_reviews_all_flags.csv", all_cleaned_df),
    "no_lemma_no_stopwords": ("cleaned_reviews_no_lemma_no_stopwords.csv", no_lemma_no_stopwords_df),
    "no_special_no_lowercase": ("cleaned_reviews_no_speical_no_lowercase.csv", no_special_no_lowercase_df)
}

In [47]:
# EXECUTE ALL DATASETS
for label, (filename, df) in datasets_to_run.items():
    print(f"\n--- Processing dataset: {label} ---")
    
    # Check if this df was indeed loaded
    if df is None or df.empty:
        print(f"Skipping {label} because it's empty or not loaded.")
        continue
        
    df_scored = df.copy().reset_index(drop=True)
    
    # Derive output filename
    base_name, ext = os.path.splitext(filename)
    out_filename = f"{base_name}_scored{ext}"
    out_path = os.path.join(OUTPUT_FOLDER, out_filename)
        
    text_items = (df_scored['review_title'].fillna('') + " \n " + df_scored['review_body'].fillna('')).tolist()
    
    # 1. Score with Gemini first
    print(f"[{label}] Scoring with Gemini...")
    df_scored['score_gemini'] = pd.NA
    for i in tqdm(range(0, len(text_items), CHUNK_SIZE), desc="Gemini"):
        chunk = text_items[i : i + CHUNK_SIZE]
        gemini_scores = get_gemini_scores(chunk)
        df_scored.iloc[i : i + len(chunk), df_scored.columns.get_loc('score_gemini')] = gemini_scores
        # Respect 15 RPM rate limit (~4 seconds per request)
        time.sleep(4.1)
        
    # 2. Score with Groq next
    print(f"[{label}] Scoring with Groq...")
    df_scored['score_groq'] = pd.NA
    for i in tqdm(range(0, len(text_items), CHUNK_SIZE), desc="Groq"):
        chunk = text_items[i : i + CHUNK_SIZE]
        groq_scores = get_groq_scores(chunk)
        df_scored.iloc[i : i + len(chunk), df_scored.columns.get_loc('score_groq')] = groq_scores
        # Respect 30 RPM rate limit (~2 seconds per request)
        time.sleep(2.1)
        
    # Save final results for this dataset once both models are done
    df_scored.to_csv(out_path, index=False)
    print(f"[{label}] Saved Gemini and Groq progression completely to {out_path}.")
    
    # Small validation
    print(f"  Valid Gemini scores: {df_scored['score_gemini'].notna().sum()}/{len(df_scored)}")
    print(f"  Valid Groq scores: {df_scored['score_groq'].notna().sum()}/{len(df_scored)}")


--- Processing dataset: all_flags ---
[all_flags] Scoring with Gemini...


Gemini: 100%|██████████| 15/15 [01:36<00:00,  6.44s/it]


[all_flags] Scoring with Groq...


Groq: 100%|██████████| 15/15 [01:35<00:00,  6.40s/it]


[all_flags] Saved Gemini and Groq progression completely to ../data/scored_data/cleaned_reviews_all_flags_scored.csv.
  Valid Gemini scores: 370/370
  Valid Groq scores: 365/370

--- Processing dataset: no_lemma_no_stopwords ---
[no_lemma_no_stopwords] Scoring with Gemini...


...

[no_lemma_no_stopwords] Saved Gemini and Groq progression completely to ../data/scored_data/cleaned_reviews_no_lemma_no_stopwords_scored.csv.
  Valid Gemini scores: 370/370
  Valid Groq scores: 366/370

--- Processing dataset: no_special_no_lowercase ---
[no_special_no_lowercase] Scoring with Gemini...


Gemini: 100%|██████████| 15/15 [01:51<00:00,  7.43s/it]


[no_special_no_lowercase] Scoring with Groq...


Groq: 100%|██████████| 15/15 [01:40<00:00,  6.68s/it]

[no_special_no_lowercase] Saved Gemini and Groq progression completely to ../data/scored_data/cleaned_reviews_no_speical_no_lowercase_scored.csv.
  Valid Gemini scores: 370/370
  Valid Groq scores: 368/370


In [48]:
# Reload and clean datasets with invalid scores
import pandas as pd
import os

# Map labels to their generated scored file paths
scored_paths = {
    label: os.path.join(OUTPUT_FOLDER, f"{os.path.splitext(filename)[0]}_scored{os.path.splitext(filename)[1]}")
    for label, (filename, _) in datasets_to_run.items()
}

# Load them
scored_dfs = {label: pd.read_csv(path) for label, path in scored_paths.items() if os.path.exists(path)}

if not scored_dfs:
    print("No scored files found. Ensure the previous cell finished successfully.")
else:
    # Identify rows with invalid scores (NaN) or missing text in ANY of the datasets
    # If a row is invalid in one, we remove it from all to keep them synced.
    indices_to_drop = set()
    
    for label, df in scored_dfs.items():
        # Invalid if Gemini or Groq failed to provide a score
        invalid_mask = df['score_gemini'].isna() | df['score_groq'].isna()
        
        # Also check for empty reviews just in case
        invalid_text = df['review_body'].isna() & df['review_title'].isna()
        
        bad_indices = df[invalid_mask | invalid_text].index
        indices_to_drop.update(bad_indices)
        print(f"[{label}] Found {len(bad_indices)} rows with invalid data/scores.")

    print(f"\nTotal unique rows to remove across all datasets: {len(indices_to_drop)}")

    # Remove and overwrite
    for label, df in scored_dfs.items():
        # Drop the indices and reset
        cleaned_df = df.drop(index=list(indices_to_drop)).reset_index(drop=True)
        
        # Save back to the same path
        out_path = scored_paths[label]
        cleaned_df.to_csv(out_path, index=False)
        print(f"[{label}] Cleaned and saved {len(cleaned_df)} rows to {out_path}.")

[all_flags] Found 6 rows with invalid data/scores.
[no_lemma_no_stopwords] Found 5 rows with invalid data/scores.
[no_special_no_lowercase] Found 3 rows with invalid data/scores.

Total unique rows to remove across all datasets: 8
[all_flags] Cleaned and saved 362 rows to ../data/scored_data/cleaned_reviews_all_flags_scored.csv.
[no_lemma_no_stopwords] Cleaned and saved 362 rows to ../data/scored_data/cleaned_reviews_no_lemma_no_stopwords_scored.csv.
[no_special_no_lowercase] Cleaned and saved 362 rows to ../data/scored_data/cleaned_reviews_no_speical_no_lowercase_scored.csv.


In [52]:
from statsmodels.stats.inter_rater import fleiss_kappa, aggregate_raters

for label, df in scored_dfs.items():
    print(f"--- Dataset: {label} ---")
    eval_df = df[['rating', 'score_gemini', 'score_groq']].dropna()
    
    coded_data = eval_df.values.astype(float).astype(int)
    
    agg_counts, _ = aggregate_raters(coded_data)
    
    kappa = fleiss_kappa(agg_counts, method='fleiss')
    
    print(f"Fleiss' Kappa score: {kappa:.4f}\n")

--- Dataset: all_flags ---
Fleiss' Kappa score: 0.5680

--- Dataset: no_lemma_no_stopwords ---
Fleiss' Kappa score: 0.5689

--- Dataset: no_special_no_lowercase ---
Fleiss' Kappa score: 0.5680



In [ ]:
import pandas as pd
import os

for label, path in scored_paths.items():
    if os.path.exists(path):
        df = pd.read_csv(path)
        
        if 'score_groq' in df.columns:
            df['score_groq'] = pd.to_numeric(df['score_groq'], errors='coerce').astype('Int64')
            
        if 'score_gemini' in df.columns:
            df['score_gemini'] = pd.to_numeric(df['score_gemini'], errors='coerce').astype('Int64')
            
        df.to_csv(path, index=False)
        print(f"[{label}] Transformed scores to integers and saved to {path}.")

[all_flags] Transformed scores to integers and saved to ../data/scored_data/cleaned_reviews_all_flags_scored.csv.
[no_lemma_no_stopwords] Transformed scores to integers and saved to ../data/scored_data/cleaned_reviews_no_lemma_no_stopwords_scored.csv.
[no_special_no_lowercase] Transformed scores to integers and saved to ../data/scored_data/cleaned_reviews_no_speical_no_lowercase_scored.csv.
